# P2 CHAIN C — answers (archive only)

Open only when marking.


## Answer — Part C1


In [ ]:
# Answer — Part C1
class Staff:
    def __init__(self, staff_id, surname, given, department):
        self.__staff_id = staff_id
        self.__surname = surname
        self.__given = given
        self.__department = department

    def get_staff_id(self):
        return self.__staff_id

    def as_dict(self):
        return {
            "staff_id": self.__staff_id,
            "surname": self.__surname,
            "given": self.__given,
            "department": self.__department,
        }

class Directory:
    def __init__(self):
        self.__people = []

    def add(self, staff):
        self.__people.append(staff)

    def find_by_id(self, staff_id):
        for p in self.__people:
            if p.get_staff_id() == staff_id:
                return p
        return None

    def all_dicts(self):
        return [p.as_dict() for p in self.__people]

roster = Directory()
for row in [
    (201, "Tan", "Mei", "Maths"),
    (202, "Lim", "Wei", "Computing"),
    (203, "Ong", "Siti", "Science"),
    (204, "Koh", "Ben", "Computing"),
    (205, "Raj", "Asha", "Languages"),
    (206, "Ng", "Hugo", "Maths"),
]:
    roster.add(Staff(*row))
print(roster.all_dicts())


## Answer — Part C2


In [ ]:
# Answer — Part C2
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["chain_c_db"]
coll = db["staff"]
coll.drop()
coll.insert_many(roster.all_dicts())
for doc in coll.find({}, {"surname": 1, "department": 1, "_id": 0}):
    print(doc)
coll.update_one({"staff_id": 202}, {"$set": {"department": "Computer Science"}})
print(coll.find_one({"staff_id": 202}, {"_id": 0}))


## Answer — Part C3


In [ ]:
# Answer — Part C3
from flask import Flask, render_template, request, redirect, url_for
from pymongo import MongoClient

app = Flask(__name__)

def get_mongo():
    client = MongoClient("mongodb://localhost:27017/")
    return client["chain_c_db"]["staff"]

@app.route("/", methods=["GET", "POST"])
def desk():
    error = None
    if request.method == "POST":
        dept_query = request.form.get("dept_query", "").strip()
        surname_query = request.form.get("surname_query", "").strip()
        if dept_query and surname_query:
            return redirect(url_for(
                "matches_view",
                dept_query=dept_query,
                surname_query=surname_query,
            ))
        error = "Please fill in all fields"
    return render_template("chain_c/desk.html", error=error)

@app.route("/matches/<dept_query>/<surname_query>")
def matches_view(dept_query, surname_query):
    people = list(get_mongo().find(
        {
            "department": {"$regex": dept_query},
            "surname": {"$regex": surname_query},
        },
        {"_id": 0},
    ))
    return render_template("chain_c/matches.html", people=people)

@app.route("/hire", methods=["GET", "POST"])
def hire():
    error = None
    if request.method == "POST":
        sid = request.form.get("sid", "").strip()
        surname = request.form.get("surname", "").strip()
        given = request.form.get("given", "").strip()
        department = request.form.get("department", "").strip()
        if not (sid and surname and given and department):
            error = "Please fill in all fields"
        else:
            get_mongo().insert_one({
                "staff_id": int(sid),
                "surname": surname,
                "given": given,
                "department": department,
            })
            return redirect(url_for("desk"))
    return render_template("chain_c/hire.html", error=error, ok=None)

if __name__ == "__main__":
    app.run(port=5002)


## Answer — Part C4


In [ ]:
# Answer — Part C4
def recv_line(sock):
    data = b""
    while b"\n" not in data:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
    return data.decode("utf-8").strip()

# server / client spines as comments — see SOCKETS.ipynb voice
